In [0]:
from pyspark.sql.functions import col, current_timestamp, round

# --- Configuration and Synchronized Paths ---
DELTA_VOLUME_PATH = "/Volumes/bc_transit_ws/gtfs/delta_volume/" 

# Static(batch) Input Paths
SILVER_ROUTES_PATH = f"{DELTA_VOLUME_PATH}silver/routes/"
SILVER_TRIPS_PATH = f"{DELTA_VOLUME_PATH}silver/trips/"


# Realtime (streaming) Input Path
SILVER_REALTIME_PATH = f"{DELTA_VOLUME_PATH}realtime_silver/vehicle_positions"

# GOLD (Streaming) Output Path
GOLD_REALTIME_PATH = f"{DELTA_VOLUME_PATH}realtime_gold/live_positions"
CHECKPOINT_LOCATION = f"{GOLD_REALTIME_PATH}/_checkpoint"

print("Starting Gold Layer Real-Time Join...")




# Load Static (Batch) Reference Data
df_routes = spark.read.format("delta").load(SILVER_ROUTES_PATH).alias("r")
df_trips = spark.read.format("delta").load(SILVER_TRIPS_PATH).alias("t")
print("Loaded static trips and routes (Batch data).")



# Load Real-Time (Streaming) Data
df_silver_realtime_stream = (
    spark.readStream
    .format("delta")
    .load(SILVER_REALTIME_PATH)
    .alias("live") # Give the stream an alias
)
print("Defined Silver Real-Time Stream (Stream data).")




# -----------------------------------------------------
# STEP 1: Stream-to-Batch Join
# -----------------------------------------------------

# 1. Join realtime positions (Stream) to static trips (Batch)
# Use 'inner' to only process real-time events that correspond to a known scheduled trip.
df_joined_trip = df_silver_realtime_stream \
        .join(
            df_trips,
            on=(col("live.trip_id") == col("t.trip_id")),
            how="inner"
        ) \
        .select(
            col("live.*"), # Keep all columns from the live stream
            col("t.route_id").alias("static_route_id"), # Get the route_id for the next join
            col("t.trip_headsign").alias("schedule_headsign") # Rename for clarity
        )

# 2. Join the result to static routes (Batch)
df_gold_stream = df_joined_trip.alias("f") \
        .join(
            df_routes,
            on=(col("f.static_route_id") == col("r.route_id")),
            how="inner" # Inner join on routes to ensure we only have valid route information
        ) \
        .select(
            # Core Real-Time Vehicle Position Data
            col("f.ingestion_timestamp"), 
            col("f.latitude"),
            col("f.longitude"),
            col("f.vehicle_id"),
            col("f.current_status_code"),
            col("f.bearing"),
            
            # Enriched Route/Trip Context (from Batch data)
            col("f.trip_id"), # Keep the ID for querying
            col("r.route_long_name").alias("route_long_name"),
            col("r.route_short_name_display").alias("route_name"),
            col("f.schedule_headsign").alias("trip_headsign"),
            
            # Audit Timestamp
            current_timestamp().alias("gold_processed_at")
        ) \
        .filter(col("route_name").isNotNull()) # Filter to ensure the enrichment succeeded


# -----------------------------------------------------
# STEP 2: Write to Gold Streaming Sink
# -----------------------------------------------------

print("\nStarting Real-time GOLD Stream Write...")

(df_gold_stream.writeStream 
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_LOCATION)
    .option("path", GOLD_REALTIME_PATH)
    .option("mergeSchema", "true") # Use this to allow schema changes
    .trigger(processingTime='10 seconds')
    .start()
)

print(f"Real-time GOLD stream is running and writing to: {GOLD_REALTIME_PATH}")
print("Monitor the stream status in the Spark UI.")

In [0]:
df_gold_stream.display()

In [0]:
# In a NEW Notebook cell:
# 1. Check the data (ensure the data is enriched with columns like route_name)
df_gold_check = spark.read.format("delta").load(GOLD_REALTIME_PATH)
df_gold_check.printSchema() # Check for the new enriched columns
df_gold_check.limit(10).display() # See the final enriched data

# 2. Check the row count
print(f"Total rows written to Gold table: {df_gold_check.count()}")

In [0]:
df_to_register = spark.read.format("delta").load(GOLD_REALTIME_PATH)

(df_to_register.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bc_transit_ws.gold.live_positions")
)
print("Registered: bc_transit_ws.gold.live_positions")